# 首次EUS-FNA/B穿刺未确诊胰腺占位患者：最终胰腺癌风险分类器

本Notebook由 `analysis_pipeline.py` 自动生成，包含数据清洗、描述性分析、四类分类器与集成模型、模型解释、图表和PPT可用结论。

In [ ]:


from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

OUT = Path.cwd()
TABLE_DIR = OUT / "tables"
FIG_DIR = OUT / "figures"
DATA_DIR = OUT / "data"
DOC_DIR = OUT / "ppt_ready_materials"

metadata = pd.read_json(DATA_DIR / "metadata.json", typ="series")
metadata


## 1. 队列定义

主分析只纳入Excel中“首次穿刺未确诊”段落且AA列最终诊断明确的病例。

In [ ]:

cohort = pd.read_csv(DATA_DIR / "analysis_cohort_first_puncture_undiagnosed.csv")
print(cohort.shape)
display(cohort.head())
display(cohort["最终诊断"].value_counts().rename(index={0:"非胰腺癌",1:"胰腺癌"}))
display(Image(filename=str(FIG_DIR / "fig_01_cohort_flow.png")))
display(Image(filename=str(FIG_DIR / "fig_02_outcome_distribution.png")))


## 2. 单因素探索

以下图表展示关键临床、检验和影像/EUS特征下最终胰腺癌比例。

In [ ]:

num_table = pd.read_csv(TABLE_DIR / "table_01_numeric_univariate.csv")
cat_table = pd.read_csv(TABLE_DIR / "table_02_categorical_univariate.csv")
display(num_table)
display(cat_table.sort_values(["变量","胰腺癌率"], ascending=[True, False]).head(25))
display(Image(filename=str(FIG_DIR / "fig_03_key_feature_cancer_rates.png")))
display(Image(filename=str(FIG_DIR / "fig_04_red_flag_risk_score.png")))


## 3. 模型设计

四类基础分类器：Logistic回归、SVM、决策树、浅层神经网络。集成方式为软投票，Logistic权重略高以保持稳定和可解释性。针型号、穿刺针数、抽吸/穿刺方法被定义为操作/取材技术变量，不进入主模型、核心解释或PPT主结论。

In [ ]:

display(Image(filename=str(FIG_DIR / "fig_05_model_workflow.png")))
excluded = pd.read_csv(TABLE_DIR / "table_00_excluded_technical_variables.csv")
display(excluded)
performance = pd.read_csv(TABLE_DIR / "table_05_model_performance_5fold_oof.csv")
repeat_summary = pd.read_csv(TABLE_DIR / "table_06_repeated_cv_summary.csv")
display(performance[["特征集","模型","AUC","平均精确率AP","敏感度","特异度","PPV","NPV","Brier"]].round(3))
display(repeat_summary)


## 4. 主模型结果

主模型使用临床+检验+影像/EUS特征，不纳入首次细胞学，也不纳入针型号、穿刺针数、抽吸/穿刺方法。

In [ ]:

display(Image(filename=str(FIG_DIR / "fig_06_roc_curves_clinical_model.png")))
display(Image(filename=str(FIG_DIR / "fig_07_precision_recall_curves.png")))
display(Image(filename=str(FIG_DIR / "fig_08_model_auc_comparison.png")))
display(Image(filename=str(FIG_DIR / "fig_09_calibration_confusion.png")))


## 5. 模型解释与临床决策

通过L1 Logistic方向、决策树规则和决策曲线辅助解释模型。

In [ ]:

l1 = pd.read_csv(TABLE_DIR / "table_08_l1_logistic_feature_direction.csv")
display(l1.head(20))
display(Image(filename=str(FIG_DIR / "fig_10_l1_feature_direction.png")))
display(Image(filename=str(FIG_DIR / "fig_11_decision_curve.png")))
display(Image(filename=str(FIG_DIR / "fig_12_decision_tree_rules.png")))


## 6. PPT可用结论

In [ ]:

md = (DOC_DIR / "PPT可用模型介绍_图表说明_分析结论.md").read_text(encoding="utf-8")
display(Markdown(md))
